### **1. Import Library**

In [10]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

### **2. Persiapan Data (Load & Merge)**

In [13]:
print("🔄 Memuat data dan shapefile...")

# Load Data Prediksi
try:
    df_prediksi = pd.read_csv("../outputs/prediksi_2024.csv")
    print("Kolom awal:", df_prediksi.columns.tolist())
    
    # --- PERBAIKAN: Rename Kolom Sesuai File CSV Kamu ---
    
    # 1. Ubah 'P1' jadi 'Aktual_2024'
    if 'P1' in df_prediksi.columns: 
        df_prediksi.rename(columns={'P1': 'Aktual_2024'}, inplace=True)
        
    # 2. Ubah 'prediksi' (huruf kecil) jadi 'Prediksi_2024'
    if 'prediksi' in df_prediksi.columns: 
        df_prediksi.rename(columns={'prediksi': 'Prediksi_2024'}, inplace=True)
        
    # 3. Jaga-jaga kalau namanya 'y_pred' (opsional)
    if 'y_pred' in df_prediksi.columns: 
        df_prediksi.rename(columns={'y_pred': 'Prediksi_2024'}, inplace=True)
        
    print("Kolom setelah rename:", df_prediksi.columns.tolist())

except:
    print("⚠️ File CSV tidak ditemukan, pastikan path benar.")

# Hitung Error jika belum ada (Sekarang pasti aman karena kolom sudah direname)
if 'Error' not in df_prediksi.columns:
    df_prediksi['Error'] = df_prediksi['Aktual_2024'] - df_prediksi['Prediksi_2024']

# Load Shapefile
path_maps = "../maps/"
filename_shp = "BATAS KABUPATEN KOTA DESEMBER 2019 DUKCAPIL.shp"
gdf = gpd.read_file(os.path.join(path_maps, filename_shp))

# Standarisasi Nama untuk Merge
def format_nama_peta(nama):
    nama = str(nama).upper().strip()
    if nama.startswith("KAB.") or nama.startswith("KAB "):
        nama = nama.replace("KAB.", "").replace("KAB ", "").strip()
        return "Kabupaten " + nama.title()
    elif nama.startswith("KOTA"):
        return nama.title()
    else:
        return "Kabupaten " + nama.title()

gdf['Wilayah_Join'] = gdf['KAB_KOTA'].apply(format_nama_peta)
df_prediksi['Wilayah_Join'] = df_prediksi['Wilayah']

# Merge
gdf_final = gdf.merge(df_prediksi, on='Wilayah_Join', how='inner')
print(f"✅ Data berhasil digabung! Jumlah wilayah: {len(gdf_final)}")

🔄 Memuat data dan shapefile...
Kolom awal: ['Wilayah', 'Tahun', 'P1', 'UHH', 'HLS', 'RLS', 'Pengeluaran', 'TPT', 'Kepadatan', 'prediksi', 'abs_error', 'ape_%']
Kolom setelah rename: ['Wilayah', 'Tahun', 'Aktual_2024', 'UHH', 'HLS', 'RLS', 'Pengeluaran', 'TPT', 'Kepadatan', 'Prediksi_2024', 'abs_error', 'ape_%']
✅ Data berhasil digabung! Jumlah wilayah: 35


### **3. Konfigurasi Peta**

In [16]:
# A. Hitung Range Skala agar Peta Aktual & Prediksi
# Ambil nilai minimum dan maksimum dari GABUNGAN kedua kolom
min_p1 = min(gdf_final['Aktual_2024'].min(), gdf_final['Prediksi_2024'].min())
max_p1 = max(gdf_final['Aktual_2024'].max(), gdf_final['Prediksi_2024'].max())

# B. Hitung Range Error agar 0 = Putih (Simetris)
max_err = max(abs(gdf_final['Error'].min()), abs(gdf_final['Error'].max()))

# C. Dictionary Konfigurasi 3 Peta
map_config = {
    'Aktual_2024': {
        'title': 'Kondisi Aktual Kemiskinan (P1) Tahun 2024',
        'cmap': 'OrRd',
        'vmin': min_p1,
        'vmax': max_p1,
        'format': '{:.2f}',
        'desc': 'Data Riil BPS (Ground Truth).\nSemakin merah = Kemiskinan semakin dalam.'
    },
    'Prediksi_2024': {
        'title': 'Prediksi Model Lasso (P1) Tahun 2024',
        'cmap': 'OrRd',
        'vmin': min_p1, 
        'vmax': max_p1,
        'format': '{:.2f}',
        'desc': 'Hasil Estimasi Model.\nBandingkan pola warnanya dengan Peta Aktual.'
    },
    'Error': {
        'title': 'Peta Sebaran Error (Residual)',
        'cmap': 'coolwarm', # Biru (Negatif) - Putih (Nol) - Merah (Positif)
        'vmin': -max_err, 
        'vmax': max_err,
        'format': '{:.2f}',
        'desc': 'Merah: Model Terlalu Rendah (Under-predict)\nBiru: Model Terlalu Tinggi (Over-predict)\nPutih: Akurat'
    }
}

path_vis = "../visualisasi_EDA/"
os.makedirs(path_vis, exist_ok=True)

print("🎨 Mulai menggambar 3 peta evaluasi...")

🎨 Mulai menggambar 3 peta evaluasi...


### **4. Looping Pembuatan Peta**

In [17]:
for col, config in map_config.items():
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    
    # --- 1. Hitung Statistik Data ---
    data_valid = gdf_final[col].dropna()
    stats = {
        'mean': data_valid.mean(),
        'min': data_valid.min(),
        'max': data_valid.max(),
        'std': data_valid.std()
    }
    
    # --- 2. Plotting Peta ---
    gdf_final.plot(
        column=col,
        ax=ax,
        legend=True,
        cmap=config['cmap'],
        vmin=config['vmin'],
        vmax=config['vmax'],
        edgecolor='black',
        linewidth=0.5,
        missing_kwds={'color': 'lightgrey', 'edgecolor': 'black', 'label': 'Data Tidak Tersedia'},
        legend_kwds={
            'label': config['title'],
            'orientation': "vertical",
            'shrink': 0.8,
            'pad': 0.05
        }
    )
    
    # --- 3. Anotasi Label Wilayah ---
    for idx, row in gdf_final.iterrows():
        if pd.notna(row.geometry):
            centroid = row.geometry.centroid
            wilayah_name = str(row['Wilayah'])
            
            # Singkat nama agar tidak penuh
            wilayah_khusus = ['Tegal', 'Semarang', 'Pekalongan', 'Magelang']
            is_khusus = any(kota in wilayah_name for kota in wilayah_khusus)
            
            if is_khusus:
                if wilayah_name.startswith('Kabupaten '):
                    wilayah_name = wilayah_name.replace('Kabupaten ', 'Kab. ')
            else:
                wilayah_name = wilayah_name.replace('Kabupaten ', '').replace('Kota ', '')
            
            # Potong jika terlalu panjang
            if len(wilayah_name) > 13:
                wilayah_name = wilayah_name[:11] + '.'
            
            ax.annotate(
                wilayah_name,
                xy=(centroid.x, centroid.y),
                fontsize=5.5,
                ha='center', va='center',
                fontweight='medium', color='black',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.6, edgecolor='none')
            )
    
    # --- 4. Elemen Visual Tambahan ---
    
    # Judul
    ax.set_title(
        f"{config['title']}\nProvinsi Jawa Tengah",
        fontsize=16, fontweight='bold', pad=20
    )
    
    # Kotak Statistik (Kiri Bawah)
    stats_text = (
        f"Statistik:\n"
        f"Rata-rata: {config['format'].format(stats['mean'])}\n"
        f"Min: {config['format'].format(stats['min'])}\n"
        f"Maksimum: {config['format'].format(stats['max'])}\n"
        f"Std Dev: {config['format'].format(stats['std'])}"
    )
    ax.text(
        0.02, 0.02, stats_text,
        transform=ax.transAxes, fontsize=9,
        verticalalignment='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='black')
    )
    
    # Kotak Deskripsi (Kanan Bawah)
    ax.text(
        0.98, 0.02, config['desc'],
        transform=ax.transAxes, fontsize=9,
        verticalalignment='bottom', horizontalalignment='right',
        style='italic',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7, edgecolor='gray')
    )
    
    # Arah Mata Angin (Kanan Atas)
    ax.text(0.95, 0.95, '↑ U', transform=ax.transAxes, fontsize=14, fontweight='bold', ha='center')
    
    # Label Sumbu (Grid)
    ax.set_xlabel('Bujur Timur', fontsize=10)
    ax.set_ylabel('Lintang Selatan', fontsize=10)
    ax.tick_params(labelsize=8)
    
    # Sumber Data (Tengah Bawah)
    ax.text(0.5, -0.12, 'Sumber: Hasil Olah Data & BPS (2024)', transform=ax.transAxes, 
            fontsize=8, ha='center', style='italic')
    
    # --- 5. Simpan Gambar ---
    file_png = os.path.join(path_vis, f"evaluasi_{col}.png")
    plt.savefig(file_png, bbox_inches='tight', dpi=300, facecolor='white')
    plt.close()
    print(f"✅ Tersimpan: {file_png}")

print("\n🎉 Selesai! Cek folder visualisasi_EDA untuk melihat hasilnya.")

✅ Tersimpan: ../visualisasi_EDA/evaluasi_Aktual_2024.png
✅ Tersimpan: ../visualisasi_EDA/evaluasi_Prediksi_2024.png
✅ Tersimpan: ../visualisasi_EDA/evaluasi_Error.png

🎉 Selesai! Cek folder visualisasi_EDA untuk melihat hasilnya.


#### **Tambahan: Membuat Peta Klasifikasi Akurasi**

In [18]:
print("🎨 Membuat Peta Klasifikasi Akurasi...")

# 1. Buat Kategori Berdasarkan Error
def kategorikan_error(err):
    # Ambang batas (threshold) bisa disesuaikan. 
    # Misal: Toleransi error 0.5 poin indeks P1.
    threshold = 0.5 
    
    if err > threshold:
        return "Under-prediction (Model < Aktual)"  # Merah (Bahaya/Anomali)
    elif err < -threshold:
        return "Over-prediction (Model > Aktual)"   # Biru (Efisien)
    else:
        return "Akurat (Sesuai Prediksi)"           # Hijau (Good Fit)

# Terapkan fungsi
gdf_final['Kategori_Akurasi'] = gdf_final['Error'].apply(kategorikan_error)

# 2. Setup Plotting Kategorial
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Definisikan warna manual agar konsisten
# Hijau = Bagus, Merah = Kurang (Under), Biru = Berlebih (Over)
color_dict = {
    'Akurat (Sesuai Prediksi)': '#abdda4',          # Hijau Lembut
    'Under-prediction (Model < Aktual)': '#d53e4f', # Merah
    'Over-prediction (Model > Aktual)': '#3288bd'   # Biru
}

# Plot Peta Kategorial
gdf_final.plot(
    column='Kategori_Akurasi',
    ax=ax,
    legend=True,
    cmap='brg', # Fallback cmap, nanti ditimpa color_dict jika mapping manual (opsional)
    # Kita pakai cara legend otomatis geopandas yang simpel:
    categorical=True,
    legend_kwds={'loc': 'lower right', 'title': 'Status Akurasi Model'},
    edgecolor='black',
    linewidth=0.5
)

# Kustomisasi Warna Manual (Opsional agar warna pasti sesuai keinginan)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#abdda4', edgecolor='black', label='Akurat (Selisih < 0.5)'),
    Patch(facecolor='#d53e4f', edgecolor='black', label='Under-prediction (Aktual > Model)'),
    Patch(facecolor='#3288bd', edgecolor='black', label='Over-prediction (Aktual < Model)')
]
# Timpa plot dengan warna manual sesuai kategori
for cat, color in color_dict.items():
    gdf_final[gdf_final['Kategori_Akurasi'] == cat].plot(
        ax=ax, color=color, edgecolor='black', linewidth=0.5
    )
ax.legend(handles=legend_elements, loc='lower right', title="Status Performansi Model")


# --- ANOTASI & LABEL (Copy dari kode sebelumnya) ---
for idx, row in gdf_final.iterrows():
    if pd.notna(row.geometry):
        centroid = row.geometry.centroid
        wilayah_name = str(row['Wilayah'])
        wilayah_khusus = ['Tegal', 'Semarang', 'Pekalongan', 'Magelang']
        is_khusus = any(kota in wilayah_name for kota in wilayah_khusus)
        
        if is_khusus:
            if wilayah_name.startswith('Kabupaten '): wilayah_name = wilayah_name.replace('Kabupaten ', 'Kab. ')
        else:
            wilayah_name = wilayah_name.replace('Kabupaten ', '').replace('Kota ', '')
        
        if len(wilayah_name) > 13: wilayah_name = wilayah_name[:11] + '.'
        
        ax.annotate(
            wilayah_name, xy=(centroid.x, centroid.y), fontsize=5.5,
            ha='center', va='center', fontweight='medium', color='black',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6, edgecolor='none')
        )

# Judul & Elemen Lain
ax.set_title("Peta Klasifikasi Akurasi Model Lasso\n(Treshold Error ±0.5)", fontsize=16, fontweight='bold', pad=20)
ax.text(0.95, 0.95, '↑ U', transform=ax.transAxes, fontsize=14, fontweight='bold', ha='center')
ax.set_axis_off()

# Simpan
output_path = "../visualisasi_EDA/peta_klasifikasi_akurasi.png"
plt.savefig(output_path, bbox_inches='tight', dpi=300)
plt.close()
print(f"✅ Peta Klasifikasi berhasil disimpan: {output_path}")

🎨 Membuat Peta Klasifikasi Akurasi...
✅ Peta Klasifikasi berhasil disimpan: ../visualisasi_EDA/peta_klasifikasi_akurasi.png
